## 1. 数据加载

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# 加载Bitcoin价格数据
data = pd.read_csv('bitcoin_second_price_MIN.csv')
print(f"数据形状: {data.shape}")
print(f"数据列: {data.columns.tolist()}")
print(f"时间范围: {data.iloc[0, 0]} 到 {data.iloc[-1, 0]}")

数据形状: (4255046, 13)
数据列: ['Unnamed: 0', 'Open Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close Time', 'Quote Asset Volume', 'Number of Trades', 'Taker Buy Base Asset Volume', 'Taker Buy Quote Asset Volume', 'Ignore']
时间范围: 0 到 4255045


In [2]:
# 数据预处理
original_columns = data.columns.tolist()
standard_columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
column_mapping = dict(zip(original_columns, standard_columns))
data = data.rename(columns=column_mapping)

# 转换时间戳
data['timestamp'] = pd.to_datetime(data['timestamp'])
data = data.set_index('timestamp')
data = data.sort_index()

print(f"处理后数据形状: {data.shape}")
print(data.head())

处理后数据形状: (4255046, 12)
                                        open     high      low    close  \
timestamp                                                                 
1970-01-01 00:00:00.000000000  1502942400000  4261.48  4261.48  4261.48   
1970-01-01 00:00:00.000000001  1502942460000  4261.48  4261.48  4261.48   
1970-01-01 00:00:00.000000002  1502942520000  4280.56  4280.56  4280.56   
1970-01-01 00:00:00.000000003  1502942580000  4261.48  4261.48  4261.48   
1970-01-01 00:00:00.000000004  1502942640000  4261.48  4261.48  4261.48   

                                volume    Volume     Close Time  \
timestamp                                                         
1970-01-01 00:00:00.000000000  4261.48  1.775183  1502942459999   
1970-01-01 00:00:00.000000001  4261.48  0.000000  1502942519999   
1970-01-01 00:00:00.000000002  4280.56  0.261074  1502942579999   
1970-01-01 00:00:00.000000003  4261.48  0.012008  1502942639999   
1970-01-01 00:00:00.000000004  4261.48  0.140796 

## 2. 交易策略基础框架

In [3]:
from abc import ABC, abstractmethod
from typing import Dict, List, Optional, Tuple, Union
from enum import Enum

class Position(Enum):
    LONG = 1
    SHORT = -1
    NEUTRAL = 0

class BaseStrategy(ABC):
    """基础策略类"""
    def __init__(self, name: str = "BaseStrategy"):
        self.name = name
        self.position = Position.NEUTRAL
        self.signals = []
        self.trades = []
        
    @abstractmethod
    def generate_signal(self, data: pd.DataFrame, index: int) -> int:
        """生成交易信号: 1=买入, -1=卖出, 0=持有"""
        pass
    
    def process_data(self, data: pd.DataFrame) -> List[int]:
        """处理整个数据集，生成信号序列"""
        self.signals = []
        for i in range(len(data)):
            signal = self.generate_signal(data, i)
            self.signals.append(signal)
        return self.signals

## 3. 具体交易策略

In [4]:
class RSIStrategy(BaseStrategy):
    """RSI策略"""
    def __init__(self, period: int = 14, upper: int = 70, lower: int = 30):
        super().__init__("RSI策略")
        self.period = period
        self.upper = upper
        self.lower = lower
        
    def calculate_rsi(self, prices: pd.Series) -> float:
        """计算RSI指标"""
        if len(prices) < self.period:
            return 50.0
        
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=self.period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=self.period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi.iloc[-1] if not pd.isna(rsi.iloc[-1]) else 50.0
        
    def generate_signal(self, data: pd.DataFrame, index: int) -> int:
        if index < self.period:
            return 0
            
        prices = data['close'].iloc[:index+1]
        rsi = self.calculate_rsi(prices)
        
        if rsi > self.upper:
            return -1  # 卖出信号
        elif rsi < self.lower:
            return 1   # 买入信号
        return 0       # 持有

In [5]:
class MACDStrategy(BaseStrategy):
    """MACD策略"""
    def __init__(self, fast: int = 12, slow: int = 26, signal: int = 9):
        super().__init__("MACD策略")
        self.fast = fast
        self.slow = slow
        self.signal_period = signal
        
    def calculate_macd(self, prices: pd.Series) -> Tuple[float, float, float]:
        """计算MACD指标"""
        if len(prices) < self.slow:
            return 0.0, 0.0, 0.0
            
        ema_fast = prices.ewm(span=self.fast).mean()
        ema_slow = prices.ewm(span=self.slow).mean()
        macd_line = ema_fast - ema_slow
        signal_line = macd_line.ewm(span=self.signal_period).mean()
        histogram = macd_line - signal_line
        
        return (
            macd_line.iloc[-1] if not pd.isna(macd_line.iloc[-1]) else 0.0,
            signal_line.iloc[-1] if not pd.isna(signal_line.iloc[-1]) else 0.0,
            histogram.iloc[-1] if not pd.isna(histogram.iloc[-1]) else 0.0
        )
        
    def generate_signal(self, data: pd.DataFrame, index: int) -> int:
        if index < self.slow:
            return 0
            
        prices = data['close'].iloc[:index+1]
        macd, signal, histogram = self.calculate_macd(prices)
        
        if macd > signal and histogram > 0:
            return 1   # 买入信号
        elif macd < signal and histogram < 0:
            return -1  # 卖出信号
        return 0       # 持有

In [6]:
class BollingerBandsStrategy(BaseStrategy):
    """布林带策略"""
    def __init__(self, period: int = 20, std_dev: float = 2.0):
        super().__init__("布林带策略")
        self.period = period
        self.std_dev = std_dev
        
    def calculate_bollinger_bands(self, prices: pd.Series) -> Tuple[float, float, float]:
        """计算布林带"""
        if len(prices) < self.period:
            return prices.iloc[-1], prices.iloc[-1], prices.iloc[-1]
            
        sma = prices.rolling(window=self.period).mean()
        std = prices.rolling(window=self.period).std()
        upper_band = sma + (std * self.std_dev)
        lower_band = sma - (std * self.std_dev)
        
        return (
            upper_band.iloc[-1] if not pd.isna(upper_band.iloc[-1]) else prices.iloc[-1],
            sma.iloc[-1] if not pd.isna(sma.iloc[-1]) else prices.iloc[-1],
            lower_band.iloc[-1] if not pd.isna(lower_band.iloc[-1]) else prices.iloc[-1]
        )
        
    def generate_signal(self, data: pd.DataFrame, index: int) -> int:
        if index < self.period:
            return 0
            
        prices = data['close'].iloc[:index+1]
        current_price = prices.iloc[-1]
        upper, middle, lower = self.calculate_bollinger_bands(prices)
        
        if current_price < lower:
            return 1   # 买入信号
        elif current_price > upper:
            return -1  # 卖出信号
        return 0       # 持有

In [7]:
class MomentumStrategy(BaseStrategy):
    """动量策略"""
    def __init__(self, lookback: int = 10, threshold: float = 0.02):
        super().__init__("动量策略")
        self.lookback = lookback
        self.threshold = threshold
        
    def generate_signal(self, data: pd.DataFrame, index: int) -> int:
        if index < self.lookback:
            return 0
            
        current_price = data['close'].iloc[index]
        past_price = data['close'].iloc[index - self.lookback]
        momentum = (current_price - past_price) / past_price
        
        if momentum > self.threshold:
            return 1   # 买入信号
        elif momentum < -self.threshold:
            return -1  # 卖出信号
        return 0       # 持有

## 4. 回测框架

In [8]:
class FractionalBacktester:
    """支持分数股交易的回测器"""
    def __init__(self, initial_capital: float = 10000, transaction_cost: float = 0.001):
        self.initial_capital = initial_capital
        self.transaction_cost = transaction_cost
        self.reset()
        
    def reset(self):
        self.capital = self.initial_capital
        self.shares = 0.0
        self.trades = []
        self.portfolio_values = []
        
    def run_backtest(self, strategy: BaseStrategy, data: pd.DataFrame) -> Dict:
        """运行回测"""
        self.reset()
        signals = strategy.process_data(data)
        
        for i, (timestamp, row) in enumerate(data.iterrows()):
            current_price = row['close']
            signal = signals[i] if i < len(signals) else 0
            
            # 计算当前组合价值
            portfolio_value = self.capital + self.shares * current_price
            self.portfolio_values.append(portfolio_value)
            
            # 执行交易
            if signal == 1 and self.capital > 0:  # 买入
                cost = self.capital * self.transaction_cost
                available_capital = self.capital - cost
                shares_to_buy = available_capital / current_price
                
                self.shares += shares_to_buy
                self.capital = 0
                
                self.trades.append({
                    'timestamp': timestamp,
                    'action': 'buy',
                    'price': current_price,
                    'shares': shares_to_buy,
                    'cost': cost
                })
                
            elif signal == -1 and self.shares > 0:  # 卖出
                proceeds = self.shares * current_price
                cost = proceeds * self.transaction_cost
                
                self.capital = proceeds - cost
                sold_shares = self.shares
                self.shares = 0.0
                
                self.trades.append({
                    'timestamp': timestamp,
                    'action': 'sell',
                    'price': current_price,
                    'shares': sold_shares,
                    'cost': cost
                })
        
        # 计算最终结果
        final_price = data['close'].iloc[-1]
        final_value = self.capital + self.shares * final_price
        
        return {
            'strategy_name': strategy.name,
            'initial_capital': self.initial_capital,
            'final_value': final_value,
            'total_return': (final_value - self.initial_capital) / self.initial_capital,
            'trades_count': len(self.trades),
            'trades': self.trades,
            'portfolio_values': self.portfolio_values
        }

## 5. 策略回测结果

In [ ]:
# 创建回测器
backtester = FractionalBacktester(initial_capital=10000)

# 定义要测试的策略
strategies = [
    RSIStrategy(period=14, upper=70, lower=30),
    MACDStrategy(fast=12, slow=26, signal=9),
    BollingerBandsStrategy(period=20, std_dev=2.0),
    MomentumStrategy(lookback=10, threshold=0.02)
]

# 存储结果
results = []

# 运行回测
for strategy in strategies:
    print(f"正在测试: {strategy.name}")
    result = backtester.run_backtest(strategy, data)
    results.append(result)
    
    print(f"  初始资金: ${result['initial_capital']:,.2f}")
    print(f"  最终价值: ${result['final_value']:,.2f}")
    print(f"  总收益率: {result['total_return']*100:.2f}%")
    print(f"  交易次数: {result['trades_count']}")
    print("-" * 50)

正在测试: RSI策略


## 6. 策略性能对比

In [ ]:
# 性能总结
print("策略性能对比:")
print("=" * 80)
print(f"{'策略名称':<20} {'总收益率':<12} {'交易次数':<8} {'最终价值':<12}")
print("-" * 80)

for result in results:
    print(f"{result['strategy_name']:<20} {result['total_return']*100:>8.2f}% "
          f"{result['trades_count']:>8} ${result['final_value']:>10,.2f}")

# 找出最佳策略
best_strategy = max(results, key=lambda x: x['total_return'])
print(f"\n最佳策略: {best_strategy['strategy_name']}")
print(f"收益率: {best_strategy['total_return']*100:.2f}%")

## 7. 买入持有基准

In [ ]:
# 计算买入持有基准
initial_price = data['close'].iloc[0]
final_price = data['close'].iloc[-1]
buy_hold_return = (final_price - initial_price) / initial_price

print(f"买入持有基准:")
print(f"  初始价格: ${initial_price:.2f}")
print(f"  最终价格: ${final_price:.2f}")
print(f"  收益率: {buy_hold_return*100:.2f}%")

print(f"\n策略vs基准对比:")
for result in results:
    excess_return = result['total_return'] - buy_hold_return
    print(f"{result['strategy_name']}: {excess_return*100:+.2f}% 超额收益")